In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [7]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents



In [8]:
# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("./Data_RAGExample")

Found 1 PDF files to process

Processing: C# 3.0 Design Patterns.pdf
  ✓ Loaded 316 pages

Total documents loaded: 316


In [9]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 4.05 for Sparc Solaris', 'creator': 'FrameMaker 5.5.6.', 'creationdate': '2007-12-06T13:21:40+00:00', 'moddate': '2008-02-10T00:26:56+03:00', 'source': 'Data_TraditionalRAG/C# 3.0 Design Patterns.pdf', 'total_pages': 316, 'page': 0, 'page_label': '1', 'source_file': 'C# 3.0 Design Patterns.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Acrobat Distiller 4.05 for Sparc Solaris', 'creator': 'FrameMaker 5.5.6.', 'creationdate': '2007-12-06T13:21:40+00:00', 'moddate': '2008-02-10T00:26:56+03:00', 'source': 'Data_TraditionalRAG/C# 3.0 Design Patterns.pdf', 'total_pages': 316, 'page': 1, 'page_label': '2', 'source_file': 'C# 3.0 Design Patterns.pdf', 'file_type': 'pdf'}, page_content='C# 3.0 Design Patterns'),
 Document(metadata={'producer': 'Acrobat Distiller 4.05 for Sparc Solaris', 'creator': 'FrameMaker 5.5.6.', 'creationdate': '2007-12-06T13:21:40+00:00', 'moddate': '2008-02-10T00:26:56+03:00', 'source': '

In [10]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [11]:
chunks=split_documents(all_pdf_documents)
chunks

Split 316 documents into 747 chunks

Example chunk:
Content: C# 3.0 Design Patterns...
Metadata: {'producer': 'Acrobat Distiller 4.05 for Sparc Solaris', 'creator': 'FrameMaker 5.5.6.', 'creationdate': '2007-12-06T13:21:40+00:00', 'moddate': '2008-02-10T00:26:56+03:00', 'source': 'Data_TraditionalRAG/C# 3.0 Design Patterns.pdf', 'total_pages': 316, 'page': 1, 'page_label': '2', 'source_file': 'C# 3.0 Design Patterns.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Acrobat Distiller 4.05 for Sparc Solaris', 'creator': 'FrameMaker 5.5.6.', 'creationdate': '2007-12-06T13:21:40+00:00', 'moddate': '2008-02-10T00:26:56+03:00', 'source': 'Data_TraditionalRAG/C# 3.0 Design Patterns.pdf', 'total_pages': 316, 'page': 1, 'page_label': '2', 'source_file': 'C# 3.0 Design Patterns.pdf', 'file_type': 'pdf'}, page_content='C# 3.0 Design Patterns'),
 Document(metadata={'producer': 'Acrobat Distiller 4.05 for Sparc Solaris', 'creator': 'FrameMaker 5.5.6.', 'creationdate': '2007-12-06T13:21:40+00:00', 'moddate': '2008-02-10T00:26:56+03:00', 'source': 'Data_TraditionalRAG/C# 3.0 Design Patterns.pdf', 'total_pages': 316, 'page': 2, 'page_label': '3', 'source_file': 'C# 3.0 Design Patterns.pdf', 'file_type': 'pdf'}, page_content='Other Microsoft .NET resources from O’Reilly\nRelated titles C# 3.0 in a Nutshell\nC# 3.0 Cookbook\nHead First C#\nHead First Design Patterns\nLearning C# 2005\nProgramming C# 3.0\n.NET Books\nResource Center

In [12]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

In [14]:
from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()

True

In [15]:
# Create embeddings and store in Chroma
vectorstore = Chroma.from_documents(documents=chunks, embedding=OpenAIEmbeddings())

In [16]:
vectorstore

In [23]:
#Similarity search
results = vectorstore.similarity_search(
    "Facade pattern",
    k=2,
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Façade Pattern | 93
Façade Pattern
Role
The role of the Façade pattern is to provide different high-level views of subsystems
whose details are hidden from users. In general, the operations that might be desir-
able from a user’s perspective could be made up of different selections of parts of the
subsystems.
Illustration
Simple interfaces to complex subsystems abound in real life. They can be created to
make frequent use of a system faster, or to differentiate between novices and power
users. A good illustration is Amazon.com’s 1-Click® system (Figure4-4), which sim-
plifies the process of ordering items for well-known customers. Normally, after
selecting an item to purchase, an Amazon customer has to enter delivery and bank
account details before the order is accepted. If these details are stored and the cus-
tomer verifies her identity in some way, 1-Click takes that customer straight to the
checkout. The customer’s stored bank account details and selected delivery address [{'crea

In [24]:
results = vectorstore.similarity_search_with_score(
    "Facade Patterns", k=1
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.272336] Façade Pattern | 93
Façade Pattern
Role
The role of the Façade pattern is to provide different high-level views of subsystems
whose details are hidden from users. In general, the operations that might be desir-
able from a user’s perspective could be made up of different selections of parts of the
subsystems.
Illustration
Simple interfaces to complex subsystems abound in real life. They can be created to
make frequent use of a system faster, or to differentiate between novices and power
users. A good illustration is Amazon.com’s 1-Click® system (Figure4-4), which sim-
plifies the process of ordering items for well-known customers. Normally, after
selecting an item to purchase, an Amazon customer has to enter delivery and bank
account details before the order is accepted. If these details are stored and the cus-
tomer verifies her identity in some way, 1-Click takes that customer straight to the
checkout. The customer’s stored bank account details and selected delivery 

In [27]:
#Search by vector
results = vectorstore.similarity_search_by_vector(
    embedding=OpenAIEmbeddings().embed_query("Facade Patterns"), k=1
)
for doc in results:
    print(f"* {doc.page_content} [{doc.metadata}]")

* Façade Pattern | 93
Façade Pattern
Role
The role of the Façade pattern is to provide different high-level views of subsystems
whose details are hidden from users. In general, the operations that might be desir-
able from a user’s perspective could be made up of different selections of parts of the
subsystems.
Illustration
Simple interfaces to complex subsystems abound in real life. They can be created to
make frequent use of a system faster, or to differentiate between novices and power
users. A good illustration is Amazon.com’s 1-Click® system (Figure4-4), which sim-
plifies the process of ordering items for well-known customers. Normally, after
selecting an item to purchase, an Amazon customer has to enter delivery and bank
account details before the order is accepted. If these details are stored and the cus-
tomer verifies her identity in some way, 1-Click takes that customer straight to the
checkout. The customer’s stored bank account details and selected delivery address [{'file

In [28]:
#A RAG agent that executes searches with a simple tool. This is a good general-purpose implementation.
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vectorstore.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [30]:
from langchain.chat_models import init_chat_model
model = init_chat_model("gpt-5.2")


In [31]:
from langchain.agents import create_agent


tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a C# 3.0 Design Patterns.pdf. "
    "Use the tool to help answer user queries."
)
agent = create_agent(model, tools, system_prompt=prompt)

In [32]:
query = (
    "What is the Facade pattern"
    "Once you get the answer, look up for implementtion example."
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the Facade patternOnce you get the answer, look up for implementtion example.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_FXnqrtQEaVUoN7YaY12p3QvH)
 Call ID: call_FXnqrtQEaVUoN7YaY12p3QvH
  Args:
    query: Facade pattern definition intent simplify interface to subsystem C# design patterns 3.0 pdf
  retrieve_context (call_AKHHF5iUC8QkjpB348PXukBJ)
 Call ID: call_AKHHF5iUC8QkjpB348PXukBJ
  Args:
    query: Facade pattern implementation example C# code design patterns 3.0 pdf
================================= Tool Message =================================
Name: retrieve_context

Source: {'producer': 'Acrobat Distiller 4.05 for Sparc Solaris', 'creator': 'FrameMaker 5.5.6.', 'total_pages': 316, 'source_file': 'C# 3.0 Design Patterns.pdf', 'page_label': '2', 'creationdate': '2007-12-06T13:21:40+00:00', 'moddate': '2008-02-10T00:

In [ ]:
#Agentic Retrieval-Augmented Generation (RAG) combines the strengths of Retrieval-Augmented Generation with agent-based reasoning. 
#Instead of retrieving documents before answering, an agent (powered by an LLM) reasons step-by-step and decides when 
#and how to retrieve information during the interaction.

